# W06B — Teacher

**Objetivo:** mostrar “orquestación ligera” (scheduler mínimo) con **timings + evidencia**.

Hoy NO metemos Docker ni Airflow. Solo Python + nuestro pipeline W06A.

## 0) Checklist

In [1]:
import os
os.chdir("..")
os.getcwd()

'c:\\Users\\hola-\\Documents\\FisicaComputacional3-Win\\REPO'

In [4]:
from pathlib import Path
import os

print("cwd:", os.getcwd())
assert Path("data/raw/pscomppars.csv").exists(), "Falta data/raw/pscomppars.csv"
assert Path("src/pipeline/w07_pipeline.py").exists(), "Falta src/pipeline/w07_pipeline.py"
assert Path("src/pipeline/w07b_runner.py").exists(), "Falta src/pipeline/w07b_runner.py"
print("OK")

cwd: c:\Users\hola-\Documents\FisicaComputacional3-Win\REPO
OK


## 1) Ejecutar el runner W07B (módulo → fallback script)

In [8]:
!ls src

__init__.py
__pycache__
ingest
pipeline
utils


In [10]:
import subprocess, sys

cmd = [sys.executable, "-m", "src.pipeline.w07b_runner"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)

if res.returncode != 0:
    print("STDERR:\n", res.stderr)
    print("\nFallback: running as script ...")
    cmd2 = [sys.executable, "src/pipeline/w07b_runner.py"]
    print("Running:", " ".join(cmd2))
    res2 = subprocess.run(cmd2, capture_output=True, text=True)
    print("STDOUT:\n", res2.stdout)
    if res2.returncode != 0:
        print("STDERR:\n", res2.stderr)
        raise RuntimeError("W07B runner failed. See stderr above.")

Running: c:\Users\hola-\Documents\FisicaComputacional3-Win\REPO\.venv\Scripts\python.exe -m src.pipeline.w07b_runner
STDOUT:
 
== Running stage: silver ==
[2026-03-01T23:33:49.529030+00:00] Stage SILVER: building silver_planet
[2026-03-01T23:33:49.617237+00:00] silver_planet rows=6081
[2026-03-01T23:33:49.617323+00:00] DONE
Result: {'mode': 'silver', 'return_code': 0, 'seconds': 0.3342}

== Running stage: dims ==
[2026-03-01T23:33:49.969223+00:00] Stage DIMS: building dim_host_full, fact_planet, dim_host_sk, fact_planet_sk
[2026-03-01T23:33:50.056799+00:00] dim_host_sk uniqueness rows=4537, keys=4537
[2026-03-01T23:33:50.056860+00:00] fact_planet rows=6081, fact_planet_sk rows=6081
[2026-03-01T23:33:50.056872+00:00] DONE
Result: {'mode': 'dims', 'return_code': 0, 'seconds': 0.4557}

== Running stage: gold ==
[2026-03-01T23:33:50.256652+00:00] Stage GOLD: building views gold_by_discoverymethod and gold_by_host
[2026-03-01T23:33:50.263957+00:00] gold views created
[2026-03-01T23:33:50.26

## 2) Verificar artifacts del runner

In [11]:
from pathlib import Path
arts = sorted([p.name for p in Path("artifacts").glob("w07b_*")])
arts

['w07b_run_report.json', 'w07b_stage_timings.csv']

## 3) Abrir el JSON del reporte y discutir métricas

In [12]:
import json
from pathlib import Path

report = json.loads(Path("artifacts/w07b_run_report.json").read_text(encoding="utf-8"))
report["stages"], report["counts"], report["artifacts"]

([{'mode': 'silver', 'return_code': 0, 'seconds': 0.3342},
  {'mode': 'dims', 'return_code': 0, 'seconds': 0.4557},
  {'mode': 'gold', 'return_code': 0, 'seconds': 0.1537},
  {'mode': 'export', 'return_code': 0, 'seconds': 0.1602}],
 {'silver_planet': 6081, 'dim_host_sk': 4537, 'fact_planet_sk': 6081},
 {'gold_by_discoverymethod': {'exists': True, 'bytes': 763},
  'gold_by_host': {'exists': True, 'bytes': 266337}})